# 🎯 LangChain Çıktı Ayrıştırıcıları (Output Parsers)

Büyük Dil Modelleri (LLM) varsayılan olarak serbest metin (unstructured text) üretir. 
Üretim ortamlarında (production), frontend veya veritabanı ile konuşabilmek için modelden **tip güvenli, doğrulanabilir ve kesin JSON/Pydantic çıktıları** almak zorundayız.

---

## 📌 İncelenen Ayrıştırıcılar:
1. **`StrOutputParser`:** Model yanıtını ham string olarak temizler.
2. **`JsonOutputParser`:** Yanıtı Python `dict` nesnesine çevirir.
3. **`PydanticOutputParser`:** Veri tiplerini (int, str, Enum) kontrol eden ve şemayı doğrulayan tip güvenli ayrıştırıcı.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model= "gpt-5.4-mini", temperature= 0)

In [ ]:
# STR Parser (default)
chain_str = ChatPromptTemplate.from_template("{soru}") | llm | StrOutputParser()
print(chain_str.invoke({"soru": "Python nedir?"}))
# String döner

In [ ]:
# JSON PARSER
json_prompt = ChatPromptTemplate.from_template("""
Aşağıdaki yorumu analiz et ve JSON olarak dön:
{{
  "sentiment" : "pozitif/negatif/nötr,
  "puan": 1-5,
  ""anahtar_kelimeler": ["..."]
}}

Yorum: {yorum}
""")

chain_json = json_prompt | llm | JsonOutputParser()

result = chain_json.invoke({
    "yorum": "Ürün harika ama kargo biraz geç geldi"
})

print(result)
print(type(result))

# {"sentiment": "pozitif","puan": 4, "anahtar_kelimeler": ["harika","kargo geç"]}

In [ ]:
# PYDANTIC PARSER - tip güvenli + validation
from langchain_core.output_parsers import PydanticOutputParser

class YorumAnalizi(BaseModel()):
    sentiment: str = Field(description= "pozitif, negatif veya nötr")
    puan: int = Field(ge= 1, le = 5,description= "1-5 arası puan")
    anahtar_kelimeler: List[str] = Field(description= "en fazla 5 kelime")
    oneri: str = Field(description= "öneri (kısa)")

parser = PydanticOutputParser(pydantic_object= YorumAnalizi)

pydantic_prompt = ChatPromptTemplate.from_template("""
Aşağıdaki yorumu analiz et

{format_instructions}

Yorum: {yorum}
""")

# format_instructions otomatik inject edilir
chain_pydantic = (
    pydantic_prompt.partial(format_instructions = parser.get_format_instructions())
    | llm
    | parser
)


result = chain_pydantic.invoke({
    "yorum": "Kalitesi güzel ama paketleme kötüydü."
})

print(result)
print(result.sentiment)
print(result.puan)

In [ ]:
# MODERN (2026): .with_structured_output() - tercih edilen yol
# Şema, modele doğrudan bildiriler; parseformat_instructions gerekmez

class Urun(BaseModel):
    ad: str
    fiyat: float
    stokta: bool

structured_llm = llm.with_structured_output(Urun)
sonuc = structured_llm.invoke("İphone 15, 54999, stokta var")

print(sonuc)

print(type(sonuc))
print(sonuc.ad, sonuc.fiyat, sonuc.stokta)

# PydanticOutputParser -> prompt'a format_instructions gömülür, cıktı parse edilir (fallback)
# .with_structured_output() -> model şemayı native döner, garantili Pydantic obje (tercih edilen)


In [ ]:
# Liste - birden fazla yorum
class YorumAnaliziListesi(BaseModel):
    yorumlar: List[YorumAnalizi]

parser_list = PydanticOutputParser(pydantic_object= YorumAnaliziListesi)

liste_prompt = ChatPromptTemplate.from_template("""

Bu yorumları ayrı ayrı analiz et ve liste olarak dön

{format_instructions}

Yorumlar: {yorumlar}
""")

chain_list = {
    liste_prompt.partial(format_instructions = parser_list.get_format_instructions())
    | llm
    | parser
}

yorumlar = """
1. Çük güzel, tavsiye ederim
2. Fiyatı uygun değil
3. Beklediğim gibi
"""

In [ ]:
# REAL USE CASE : Review extractor (production-ready)
class UrunReview(BaseModel):
    urun_adi : str = Field(description= "Bahsedilen ürün adı")
    fiyat_tahmin: bool = Field(description= "Fiyattan memnun mu?")
    kalite_skoru: int = Field(ge= 1, le = 10)
    kargo_problemi: bool = Field(description= "Kargo sorunu var mı?")
    pozitif_yonler: List[str]
    negatif_yonler: List[str]


parser_review = PydanticOutputParser(pydantic_object= UrunReview)

review_prompt = ChatPromptTemplate.from_template("""

Aşağıdaki ürün yorumunu detaylı analiz et.

{format_instructions}

Yorum: {yorum}
""")

chain_review = (
    review_prompt.partial(format_instruction = parser_review.get_format_instructions())
    | llm
    | parser_review
)
test_yorum = """

Bu bluetooh kulaklığı 890 TL2ye aldım. Ses kalitesi gerçekten çok iyi,
bass etkileyici. Ancak kargoda 1 gün gecikme oldu ve kutusu biraz ezikti.
Pil ömrü 8 saat civarı, iyi sayılır. Fiyat biraz pahalı geldi ama kalitesi
bunu haklı çıkarıyor. Mikrofonu da net, görüşmelerde sorun yok.
"""

result = chain_review.invoke({"yorum": test_yorum})
print(result.model_dump_json(indent = 2))

